In [6]:
import json
import os
from pathlib import Path
from collections import Counter
import pandas as pd

json_dir = Path("addmypet")

entity_counter = Counter()
species_count = 0
parse_errors = []

for json_file in sorted(json_dir.glob("*.json")):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            raw = f.read().replace(": NaN", ": null").replace(":NaN", ":null")
            record = json.loads(raw)
        species_count += 1
        for entry in record.get("data", []):
            if isinstance(entry, dict) and "entity" in entry:
                entity_counter[entry["entity"]] += 1
    except Exception as e:
        parse_errors.append((json_file.name, str(e)))

print(f"Species files processed : {species_count}")
print(f"Unique entities found   : {len(entity_counter)}")
print(f"Total entity occurrences: {sum(entity_counter.values())}")
if parse_errors:
    print(f"\nFiles with parse errors ({len(parse_errors)}):")
    for fname, err in parse_errors:
        print(f"  {fname}: {err}")

df_entities = pd.DataFrame(entity_counter.most_common(), columns=["entity", "occurrences"])

pd.set_option("display.max_rows", None)
print("\nEntity statistics (sorted by occurrence):")
display(df_entities)


Species files processed : 7281
Unique entities found   : 705
Total entity occurrences: 61173

Entity statistics (sorted by occurrence):


,entity,occurrences
0,am,7221
1,Wwi,6652
2,Ri,6079
3,Wwb,5955
4,Li,4315
5,ab,4089
6,tp,3950
7,Lp,3915
8,tx,2940
9,Wwp,2415


In [7]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd

# ── Canonical DEB zero-variate entities ───────────────────────────────────────
CANONICAL = {
    # ages / time intervals
    'ab', 'ah', 'aj', 'am', 'ap', 'as', 'ae', 'ax', 'aR',
    'tg', 'tx', 'tR', 'ts', 'tb', 'te', 'tj', 'tp', 'tW',
    # lengths
    'L0', 'Lh', 'Lb', 'Lx', 'Ls', 'Lj', 'Lp', 'LR', 'Li',
    'Lr', 'Lo', 'Lam', 'LsWw', 'LtWw',
    'dLb', 'dLp', 'dLj',
    # volumes
    'V0', 'Vb', 'Vp', 'Vi',
    # wet weights
    'Ww0', 'Wwh', 'Wwb', 'Wwx', 'Wws', 'Wwj', 'Wwp', 'Wwe', 'Wwi',
    'WwR', 'Ww_L', 'dWw', 'dWwb',
    # dry weights
    'Wd0', 'Wdh', 'Wdb', 'Wdx', 'Wds', 'Wdj', 'Wdp', 'Wde', 'Wdi', 'Wdr',
    'WdR',
    # carbon / nitrogen content
    'WC0', 'WCb', 'WCi', 'WCp', 'WCx', 'WN0', 'WNb',
    # energy
    'E0', 'Eb', 'Eh', 'Ej', 'EXx', 'EL',
    # rates / reproduction
    'Ri', 'rB', 'GSI', 'Ni',
    # feeding / respiration
    'Fm', 'K', 'KV',
    'JOi', 'JO', 'JOw',
    'JX', 'JXi', 'JXm', 'JX_L',
    'JCi',
    'pAm', 'pXi', 'pXm', 'pL', 'pP', 'pA',
    # other scalars
    'sM', 'sgr', 'xi_WE', 'RQ', 'NR',
    'del_S', 'del_Lip',
    'k_tar',
}

# Sort by length descending so the longest canonical prefix wins
CANONICAL_SORTED = sorted(CANONICAL, key=len, reverse=True)

# ── MANUAL: cases that cannot be resolved by prefix matching ──────────────────
MANUAL = {
    # JC prefix not in CANONICAL — map to JCi
    'JC1': 'JCi', 'JC2': 'JCi', 'JC3': 'JCi', 'JCe': 'JCi', 'JC_p': 'JCi',
    # pX not in CANONICAL — map to pXi
    'pX': 'pXi', 'pX25': 'pXi',
    # p_Xm: underscore is a prefix not suffix — map to pXm
    'p_Xm': 'pXm',
    # NR is in CANONICAL but biologically maps to Ni
    'NR': 'Ni',
    # case change: lowercase c → uppercase C
    'Wc0': 'WC0',
}


def to_canonical(raw):
    """Return the canonical entity name, or None if unmappable."""
    raw = raw.strip()
    if raw in CANONICAL:
        return raw
    if raw in MANUAL:
        return MANUAL[raw]
    for c in CANONICAL_SORTED:
        if raw.startswith(c):
            return c
    return None


# ── Step 1: count occurrences by canonical entity ─────────────────────────────
json_dir = Path("addmypet")

canon_counter = Counter()
unmapped      = Counter()

for json_file in sorted(json_dir.glob("*.json")):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            raw = f.read().replace(": NaN", ": null").replace(":NaN", ":null")
            record = json.loads(raw)
        for entry in record.get("data", []):
            if isinstance(entry, dict) and "entity" in entry:
                e = entry["entity"]
                c = to_canonical(e)
                if c:
                    canon_counter[c] += 1
                else:
                    unmapped[e] += 1
    except Exception as ex:
        print(f"Parse error in {json_file.name}: {ex}")

# ── Step 2: statistics ────────────────────────────────────────────────────────
df_canon = pd.DataFrame(canon_counter.most_common(), columns=["CANONICAL", "occurrences"])
print(f"Canonical entities : {len(df_canon)}")
print(f"Total occurrences  : {df_canon['occurrences'].sum()}")
if unmapped:
    print(f"\nNot mapped — {len(unmapped)} labels, {sum(unmapped.values())} occurrences:")
    for e, cnt in unmapped.most_common():
        print(f"  {e:30s}  {cnt}")

pd.set_option("display.max_rows", None)
print("\nCanonical entity statistics:")
display(df_canon)

# ── Step 3: rewrite JSON files with canonical entity names ───────────────────
updated_files = 0
total_files   = len(list(json_dir.glob("*.json")))

for json_file in sorted(json_dir.glob("*.json")):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            raw_text = f.read().replace(": NaN", ": null").replace(":NaN", ":null")
            record = json.loads(raw_text)
    except Exception as ex:
        print(f"Skipping {json_file.name}: {ex}")
        continue

    changed = False
    for entry in record.get("data", []):
        if isinstance(entry, dict) and "entity" in entry:
            c = to_canonical(entry["entity"])
            if c and c != entry["entity"]:
                entry["entity"] = c
                changed = True

    if changed:
        with open(json_file, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        updated_files += 1

print(f"\nJSON files updated: {updated_files} / {total_files}")


Canonical entities : 99
Total occurrences  : 60752

Not mapped — 246 labels, 421 occurrences:
  L2                              13
  L1                              12
  L3                              11
  t1                              10
  t2                              10
  t3                              10
  t4                              8
  Ww1                             7
  RL                              7
  Ww2                             7
  L4                              6
  t5                              5
  R20                             5
  Le                              4
  L5                              4
  R7                              4
  N18                             4
  Ww3                             4
  L6                              3
  L18                             3
  L365                            3
  WwL                             3
  N26                             3
  Ww4                             3
  Ww5                             3


,CANONICAL,occurrences
0,Wwi,8155
1,am,7255
2,Ri,6107
3,Wwb,5977
4,Li,5058
5,Lp,4426
6,tp,4404
7,ab,4232
8,tx,2941
9,Wwp,2516



JSON files updated: 2223 / 7281


In [8]:
import json
import os
from pathlib import Path
from collections import Counter
import pandas as pd

json_dir = Path("addmypet")

entity_counter = Counter()
species_count = 0
parse_errors = []

for json_file in sorted(json_dir.glob("*.json")):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            raw = f.read().replace(": NaN", ": null").replace(":NaN", ":null")
            record = json.loads(raw)
        species_count += 1
        for entry in record.get("data", []):
            if isinstance(entry, dict) and "entity" in entry:
                entity_counter[entry["entity"]] += 1
    except Exception as e:
        parse_errors.append((json_file.name, str(e)))

print(f"Species files processed : {species_count}")
print(f"Unique entities found   : {len(entity_counter)}")
print(f"Total entity occurrences: {sum(entity_counter.values())}")
if parse_errors:
    print(f"\nFiles with parse errors ({len(parse_errors)}):")
    for fname, err in parse_errors:
        print(f"  {fname}: {err}")

df_entities = pd.DataFrame(entity_counter.most_common(), columns=["entity", "occurrences"])

pd.set_option("display.max_rows", None)
print("\nEntity statistics (sorted by occurrence):")
display(df_entities)


Species files processed : 7281
Unique entities found   : 345
Total entity occurrences: 61173

Entity statistics (sorted by occurrence):


,entity,occurrences
0,Wwi,8155
1,am,7255
2,Ri,6107
3,Wwb,5977
4,Li,5058
5,Lp,4426
6,tp,4404
7,ab,4232
8,tx,2941
9,Wwp,2516
